# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

## 1. Setup
Imports, display options, and repository paths.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 50)

ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

print(f"Setup complete. Paper: {MY_PAPER}")

Setup complete. Paper: card


### 2.1 Data
NLSYM 1976 cross-section (Card 1995). Sample: men with valid 1976 wage and education, N = 3,010 (Table 1, col. 3).

In [ ]:
paths = fetch(MY_PAPER)
raw = pd.read_stata(paths[0])

df = raw.copy()
df["agesq"]   = df["age"] ** 2
df["expersq"] = df["exper"] ** 2

NEEDED = ["lwage", "educ", "exper", "expersq", "age", "agesq", "nearc4",
          "black", "south", "smsa", "smsa66"] + [f"reg66{i}" for i in range(1, 9)]
df = df.dropna(subset=NEEDED).copy()
assert len(df) == 3010, f"N = {len(df)}, expected 3,010"

print("exper == age − educ − 6:", (df["exper"] == df["age"] - df["educ"] - 6).mean().round(4))
display(df[["lwage", "educ"]].agg(["mean", "std"]).T)

  data/card.dta  0.5 MB
exper == age − educ − 6: 1.0


,mean,std
lwage,6.2618,0.4438
educ,13.2635,2.6769


### 2.2 Specification
Card (1995) Table 2 col. 2 (OLS) and Table 3 Panel A col. 5 (2SLS). Controls: experience and its square (treated as exogenous), black, South and SMSA in 1976, SMSA in 1966, and eight 1966 region dummies. `nearc4` (a 4-year college in the local labour market in 1966) instruments education. Conventional standard errors, as in the paper.

In [ ]:
EXOG     = "black + south + smsa + smsa66 + " + " + ".join(f"reg66{i}" for i in range(1, 9))
CONTROLS = f"exper + expersq + {EXOG}"

# Panel A -> experience exogenous
ols_fit = smf.ols(f"lwage ~ educ + {CONTROLS}", data=df).fit()
fs_fit  = smf.ols(f"educ ~ nearc4 + {CONTROLS}", data=df).fit()
rf_fit  = smf.ols(f"lwage ~ nearc4 + {CONTROLS}", data=df).fit()
iv_fit  = IV2SLS.from_formula(f"lwage ~ 1 + {CONTROLS} + [educ ~ nearc4]",
                              data=df).fit(cov_type="unadjusted", debiased=True)

# Panel B -> experience endogenous | instrumented by age and age²
fsB_fit = smf.ols(f"educ ~ nearc4 + age + agesq + {EXOG}", data=df).fit()
rfB_fit = smf.ols(f"lwage ~ nearc4 + age + agesq + {EXOG}", data=df).fit()
ivB_fit = IV2SLS.from_formula(
    f"lwage ~ 1 + {EXOG} + [educ + exper + expersq ~ nearc4 + age + agesq]",
    data=df).fit(cov_type="unadjusted", debiased=True)

### 2.3 Comparison
Card's published estimates beside this notebook's, no family background controls.

In [7]:
rows = [
    ("T2 (2): OLS, educ",              0.075, 0.003, ols_fit.params["educ"],   ols_fit.bse["educ"]),
    ("T3A (1): first stage, nearc4",   0.320, 0.088, fs_fit.params["nearc4"],  fs_fit.bse["nearc4"]),
    ("T3A (3): reduced form, nearc4",  0.042, 0.018, rf_fit.params["nearc4"],  rf_fit.bse["nearc4"]),
    ("T3A (5): 2SLS, educ",            0.132, 0.055, iv_fit.params["educ"],    iv_fit.std_errors["educ"]),
    ("T3B (1): first stage, nearc4",   0.382, 0.114, fsB_fit.params["nearc4"], fsB_fit.bse["nearc4"]),
    ("T3B (3): reduced form, nearc4",  0.047, 0.019, rfB_fit.params["nearc4"], rfB_fit.bse["nearc4"]),
    ("T3B (5): 2SLS, educ",            0.122, 0.046, ivB_fit.params["educ"],   ivB_fit.std_errors["educ"]),
]
comp = pd.DataFrame(rows, columns=["Estimate", "Card β", "Card SE", "Mine β", "Mine SE"])
comp["Δβ"] = comp["Mine β"] - comp["Card β"]
comp.to_csv(ROOT / "output" / "replication_table.csv", index=False)
display(comp.round(3))

,Estimate,Card β,Card SE,Mine β,Mine SE,Δβ
0,"T2 (2): OLS, educ",0.0750,0.0030,0.0750,0.0030,-0.0000
1,"T3A (1): first stage, nearc4",0.3200,0.0880,0.3200,0.0880,-0.0000
2,"T3A (3): reduced form, nearc4",0.0420,0.0180,0.0420,0.0180,0.0000
3,"T3A (5): 2SLS, educ",0.1320,0.0550,0.1320,0.0550,-0.0000
4,"T3B (1): first stage, nearc4",0.3820,0.1140,0.3920,0.1140,0.0100
5,"T3B (3): reduced form, nearc4",0.0470,0.0190,0.0320,0.0170,-0.0150
6,"T3B (5): 2SLS, educ",0.1220,0.0460,0.1220,0.0460,0.0000


### 2.4 Unit check
Confirms the outcome is log hourly wages, that 2SLS equals the Wald ratio (Card fn. 25), and converts the estimate into a percentage.

In [8]:
print("lwage == log(wage):", np.allclose(np.log(df["wage"]), df["lwage"]))

wald = rf_fit.params["nearc4"] / fs_fit.params["nearc4"]
print(f"Wald ratio {wald:.4f} vs 2SLS {iv_fit.params['educ']:.4f}")

b = iv_fit.params["educ"]
print(f"2SLS: {b:.3f} log points per year = {100*(np.exp(b)-1):.1f}% higher hourly wage per extra year")
print(f"First-stage F (nearc4): {fs_fit.tvalues['nearc4']**2:.2f}")

lwage == log(wage): True
Wald ratio 0.1315 vs 2SLS 0.1315
2SLS: 0.132 log points per year = 14.1% higher hourly wage per extra year
First-stage F (nearc4): 13.26


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. Carneiro & Heckman (2002), *Economic Journal* 112(482). IV > OLS is not evidence of credit constraints.
2. Kitagawa (2015), *Econometrica* 83(5). Rejects proximity IV validity without covariates.
3. Kling (2001), *JBES* 19(3). 2SLS weights heterogeneous returns, not the population average.

**The identification strategy and its assumption, in my own words:**

Card compares men who grew up in a local labour market with a 4-year college in 1966 to men who did not. Living near a college makes it cheaper to attend, since students can live at home with their families, so proximity raises schooling: by about 0.32 years in the first stage (F ≈ 13).

The key assumption is exclusion: conditional on race, region, and urban residence, growing up near a college affects 1976 wages only through the extra schooling it causes. If families that value education choose to live near colleges, or if college towns have better schools or higher wages, this fails and the estimate is biased upward.

Because returns likely differ across people, the 2SLS estimate is a local average treatment effect: the return for compliers, men whose schooling depended on having a college nearby. Card's Figure 1 suggests these are mostly men from low-education families, so 0.132 need not apply to the average worker.

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [11]:
EXOG     = "black + south + smsa + smsa66 + " + " + ".join(f"reg66{i}" for i in range(1, 9))
CONTROLS = f"exper + expersq + {EXOG}"

# Table 2 col 2 and Table 3 Panel A -> experience exogenous
ols_fit = smf.ols(f"lwage ~ educ + {CONTROLS}", data=df).fit()
fs_fit  = smf.ols(f"educ ~ nearc4 + {CONTROLS}", data=df).fit()
rf_fit  = smf.ols(f"lwage ~ nearc4 + {CONTROLS}", data=df).fit()
iv_fit  = IV2SLS.from_formula(f"lwage ~ 1 + {CONTROLS} + [educ ~ nearc4]",
                              data=df).fit(cov_type="unadjusted", debiased=True)

# Table 3 Panel B -> experience instrumented by age, age²
fsB_fit = smf.ols(f"educ ~ nearc4 + age + agesq + {EXOG}", data=df).fit()
rfB_fit = smf.ols(f"lwage ~ nearc4 + age + agesq + {EXOG}", data=df).fit()
ivB_fit = IV2SLS.from_formula(
    f"lwage ~ 1 + {EXOG} + [educ + exper + expersq ~ nearc4 + age + agesq]",
    data=df).fit(cov_type="unadjusted", debiased=True)

print(f"OLS {ols_fit.params['educ']:.3f} | 2SLS {iv_fit.params['educ']:.3f} | first-stage F {fs_fit.tvalues['nearc4']**2:.1f}")

OLS 0.075 | 2SLS 0.132 | first-stage F 13.3


### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [ ]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)

headline.loc["Table 2, col 2: OLS return to schooling"]         = [0.075, ols_fit.params["educ"],   np.nan, "Match; gap is rounding of published 3dp"]
headline.loc["Table 3A, col 1: first stage, nearc4 on educ"]    = [0.320, fs_fit.params["nearc4"],  np.nan, "Match; rounding"]
headline.loc["Table 3A, col 3: reduced form, nearc4 on lwage"]  = [0.042, rf_fit.params["nearc4"],  np.nan, "Match; rounding"]
headline.loc["Table 3A, col 5: 2SLS return to schooling"]       = [0.132, iv_fit.params["educ"],    np.nan, "Match; rounding"]
headline.loc["Table 3B, col 1: first stage, nearc4 on educ"]    = [0.382, fsB_fit.params["nearc4"], np.nan, "Probable printing error: identical SE (0.114)"]
headline.loc["Table 3B, col 3: reduced form, nearc4 on lwage"]  = [0.047, rfB_fit.params["nearc4"], np.nan, "Card's column matches FS x 2SLS, not a regression; see log row 3"]
headline.loc["Table 3B, col 5: 2SLS return to schooling"]       = [0.122, ivB_fit.params["educ"],   np.nan, "Match; rounding"]


headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

,paper_estimate,my_estimate,gap_pct,likely_source
estimate,,,,
"Table 2, col 2: OLS return to schooling",0.0750,0.0747,-0.4090,Match; gap is rounding of published 3dp
"Table 3A, col 1: first stage, nearc4 on educ",0.3200,0.3199,-0.0316,Match; rounding
"Table 3A, col 3: reduced form, nearc4 on lwage",0.0420,0.0421,0.1617,Match; rounding
"Table 3A, col 5: 2SLS return to schooling",0.1320,0.1315,-0.3759,Match; rounding
"Table 3B, col 1: first stage, nearc4 on educ",0.3820,0.3924,2.7178,Probable printing error: identical SE (0.114)
"Table 3B, col 3: reduced form, nearc4 on lwage",0.0470,0.0325,-30.9468,Paper's RF definition unclear; see discrepancy...
"Table 3B, col 5: 2SLS return to schooling",0.1220,0.1224,0.3194,Match; rounding


#### Investigating discrepancy 3: Table 3B reduced form
Tests two explanations for the gap (0.032 vs 0.047): (a) Card's column is the implied schooling channel, first stage × 2SLS, rather than a plain regression; (b) Card used a different age function than age + age².

In [13]:
print(f"FS × IV: {fsB_fit.params['nearc4'] * ivB_fit.params['educ']:.4f} | Card RF: 0.047")
for label, age_terms in [("age dummies", "C(age)"), ("cubic in age", "age + agesq + I(age**3)")]:
    fs = smf.ols(f"educ ~ nearc4 + {age_terms} + {EXOG}", df).fit()
    rf = smf.ols(f"lwage ~ nearc4 + {age_terms} + {EXOG}", df).fit()
    print(f"{label}: FS {fs.params['nearc4']:.3f} ({fs.bse['nearc4']:.3f}) | "
          f"RF {rf.params['nearc4']:.3f} ({rf.bse['nearc4']:.3f})")

FS × IV: 0.0480 | Card RF: 0.047
age dummies: FS 0.388 (0.114) | RF 0.031 (0.017)
cubic in age: FS 0.392 (0.114) | RF 0.033 (0.017)


### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | Table 2 col 2; Table 3A cols 1, 3, 5; Table 3B col 5 | Unrounded vs published 3dp | < 1% | Rounding | SEs also match to 3dp; N = 3,010; lwage and educ means/SDs match Table 3 notes; 2SLS equals Wald ratio RF/FS | No |
| 2 | Table 3B, col 1: first stage | 0.392 vs 0.382 | +0.010 (+2.7%) | Probable printing error | SE identical (0.114); age dummies (0.388) and cubic in age (0.392) do not reproduce 0.382, so the age function is ruled out; Card's text (p. 13) cites "0.32 to 0.38" | Partly: text and table may both carry the same value |
| 3 | Table 3B, col 3: reduced form | 0.032 (0.017) vs 0.047 (0.019) | −0.015 (−30.9%) | Undocumented choice | Plain RF is robust to age function (0.031–0.033); but FS × 2SLS = 0.048 ≈ 0.047, so Card's column is the implied schooling channel. Holding age fixed, nearc4 also lowers experience (exper = age − educ − 6), which a plain RF absorbs. Source of Card's SE (0.019) unknown | Yes: column is labelled a reduced form but matches a derived quantity |
| 4 | Table 2 experience² | −0.229 vs ≈ −0.00229 | Factor of 100 | Variable definition | Card's row label reads "Experience-Squared /100" | No |
| 5 | Family-background columns (Table 2 cols 3–5, Table 3 even cols) | Not replicated | n/a | Variable definition | Card's 14 variables include 8 parental-education class dummies and mean imputation (p. 4); class cut-offs not given, not in card.dta | Yes: class definitions not reported |

**Ambiguities in the paper, to take into Section 3:**

- Experience: Panel A treats it as exogenous, Panel B instruments it with age. Card reports both without picking a headline (0.132 vs 0.122).
- Reduced form under endogenous experience: Table 3B col 3 is labelled a reduced form, but matches first stage × 2SLS rather than a regression (row 3).
- Family background: parental-education classes and imputation are described only loosely.
- Standard errors: conventional, never stated; no robust SEs reported.
- Proximity timing: measured at 1966 residence, not at the age of the college decision (pp. 16–17).
- Region and urban controls mix years: region dummies and SMSA are from 1966, while South and SMSA are also included for 1976. The paper doesn't justify the mix.